In [2]:
from biomass import *

from const import path_scenario_data, SCENARIO

SyntaxError: parameter without a default follows parameter with a default (biomass.py, line 152)

In [ ]:
# Crop consumption in Gg dm/yr, per type of use and crop type including other crops. Dimensions:  [5,17,27] (t) , [NUFPT, NFCT, NRT](time)
crops_cons = read_mym_df(path_scenario_data + f'/AGRCONSCTF_DM.OUT').set_index(["time", "DIM_1", "DIM_2"])  

# Wood demand per woodtype in 1000m3/yr. Dimensions: [4,27](t), [NWCT,NRT] (time)
wood_demand = read_mym_df(path_scenario_data + 'WDEMAND.OUT').set_index(["time", "DIM_1"])

# Feed consumption per grazing system type, feed product type and animal type in Gg dm/yr Dimensions:  [3,6,6,27] (t), [NGST,NFPT,NAT,NRT] (t)
feed_cons = read_mym_df(path_scenario_data + 'TFEED.OUT').set_index(["time", "DIM_1", "DIM_2", "DIM_3"])

# Animal products  Unit=Gg dm/yr; Label=Consumption of animal products in dry matter, per type of use and animal type [5,6,27] (t) [NUFPT,NAPT,NRT] 
animal_products_cons = read_mym_df(path_scenario_data + 'AGRCONSA_DM.OUT').set_index(["time", "DIM_1", "DIM_2"])

# Biofuel crops production (same as consumption) difference is only made in energy trade, not for actual crops calculation
# Unit= Gg dm/yr; Label= Production of biofuels (dry matter) [5,27] [NBCT,NRT] (t)
biofuel_crops = read_mym_df(path_scenario_data + 'AGRPRODBF_dm.OUT').set_index(["time", "DIM_1"])

# Materials buildings (BUMA output) in kt
buildings_mat = pd.read_csv(path_input_data + f'IMAGE_MAT_out/{SCENARIO}/material_output_buma_RASMI.csv', header = 0).set_index(['Unnamed: 0', 'flow', 'type', 'area', 'material'])

# Split up different biomass types 
# Crops: Split up crops

splitted_up_crops_total = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                            selected_DIM1= 'total', biomass_type = 'crops')
splitted_up_crops_food = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                        selected_DIM1= 'food', biomass_type = 'crops')
splitted_up_crops_feed = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                        selected_DIM1= 'feed', biomass_type = 'crops')
splitted_up_crops_other_use = split_up_crops(crops_cons, gg_to_bt, drop_dim = ['DIM_1', 'DIM_2'], 
                                        selected_DIM1= 'other use', biomass_type = 'crops')
# Feed
splitted_up_feed = split_up_feed(feed_cons)

# Wood & wood from buildings
splitted_up_wood = split_up_wood(wood_demand, buildings_mat)

# Animal Products
splitted_up_animal_products = split_up_animal_products(animal_products_cons)

# Biofuel crops 
splitted_up_biofuel_crops = split_up_biofuelcrops(biofuel_crops)

#%% get global values

crops_total_consumption_global = sum_by_region(splitted_up_crops_total)
feed_total_consumption_global = sum_by_region(splitted_up_feed)

# make sankey
sankey_total, link_source_bio, link_target_bio, link_value_bio  = sankey_total_biomass(2060, 27)

TypeError: split_up_wood() got an unexpected keyword argument 'buildings_mat'